# AbbVie Inc. — Financial Performance Analysis 2020–2023
**Author:** Darius Nobles  
**Tools:** Python · Pandas · Matplotlib · NumPy  
**Data Source:** AbbVie Inc. Annual Reports & SEC 10-K Filings (2020–2023)  

---

## Business Problem

AbbVie is a global biopharmaceutical company with over $55B in annual revenue. In 2020, it completed one of the largest acquisitions in pharma history — buying Allergan for $63B. This analysis answers:

> **How has AbbVie's financial performance, profitability, and balance sheet strength evolved across 4 fiscal years — and what does that signal about its strategic trajectory?**

This is the kind of analysis a Financial Analyst, Business Analyst, or BI Analyst would produce for an investment committee or executive leadership team.

---

## Table of Contents
1. [Data Loading & Structure](#1)
2. [Revenue & Growth Analysis](#2)
3. [Profitability & Margin Analysis](#3)
4. [R&D & Operating Expense Analysis](#4)
5. [Cash Flow & Shareholder Returns](#5)
6. [Debt & Balance Sheet Analysis](#6)
7. [Key Financial Ratios](#7)
8. [Horizontal Analysis](#8)
9. [Executive Summary & Recommendations](#9)

---
## Section 1: Data Loading & Structure <a id='1'></a>

In [ ]:
# ── Import libraries ──
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import json
import warnings
warnings.filterwarnings('ignore')

# ── Chart styling ──
plt.rcParams.update({
    'figure.dpi': 150,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.facecolor': '#F8FAFC',
    'grid.alpha': 0.4,
    'grid.linestyle': '--',
})

BLUE='#2C6FBF'; GOLD='#F0B429'; GREEN='#10B981'
RED='#F43F5E'; PURPLE='#7C3AED'; ORANGE='#F97316'

print('Libraries loaded.')

In [ ]:
# ── Load financial data from SEC 10-K filings ──
with open('data/abbvie_financials.json') as f:
    data = json.load(f)

IS = data['income_statement']
BS = data['balance_sheet']
CF = data['cash_flow']

YEARS = IS['years']

# ── Build a clean summary DataFrame ──
df = pd.DataFrame({
    'Year':             YEARS,
    'Revenue ($M)':     IS['revenue'],
    'Gross Profit ($M)':IS['gross_profit'],
    'EBITDA ($M)':      IS['ebitda'],
    'Net Income ($M)':  IS['net_income'],
    'Free Cash Flow':   CF['free_cash_flow'],
    'Total Debt ($M)':  BS['total_debt'],
}).set_index('Year')

print('=== AbbVie Financial Summary ($M) ===')
print(df.to_string())

---
## Section 2: Revenue & Growth Analysis <a id='2'></a>

In [ ]:
# ── Calculate Year-over-Year growth rates ──
revenue = IS['revenue']
yoy_growth = [None] + [round((revenue[i]-revenue[i-1])/revenue[i-1]*100, 2)
                        for i in range(1, len(revenue))]

print('=== Revenue Growth Analysis ===')
for year, rev, growth in zip(YEARS, revenue, yoy_growth):
    g_str = f'{growth:+.1f}%' if growth else 'Baseline'
    print(f'{year}: ${rev/1000:.2f}B  |  YoY: {g_str}')

cagr = (revenue[-1]/revenue[0])**(1/(len(revenue)-1)) - 1
print(f'\n3-Year CAGR (2020→2023): {cagr*100:.1f}%')

In [ ]:
# ── Chart: Revenue trend with YoY growth overlay ──
fig, ax1 = plt.subplots(figsize=(10, 6))
ax2 = ax1.twinx()

bars = ax1.bar(YEARS, revenue, color=[BLUE]*3+[GOLD],
               edgecolor='none', width=0.55, alpha=0.88)
for bar, val in zip(bars, revenue):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+400,
             f'${val/1000:.1f}B', ha='center', fontsize=10.5, fontweight='bold')

valid_yoy = [(y, g) for y, g in zip(YEARS[1:], yoy_growth[1:]) if g is not None]
yrs, grws = zip(*valid_yoy)
ax2.plot(yrs, grws, color=GREEN, linewidth=2.5, marker='D',
         markersize=8, markerfacecolor='white', markeredgewidth=2.5)
for x, g in zip(yrs, grws):
    col = GREEN if g > 0 else RED
    ax2.annotate(f'{g:+.1f}%', (x, g),
                 textcoords='offset points', xytext=(0, 12),
                 ha='center', fontsize=9, color=col, fontweight='bold')

ax1.set_ylabel('Revenue (USD)', fontsize=11)
ax2.set_ylabel('YoY Growth (%)', fontsize=11, color=GREEN)
ax2.tick_params(colors=GREEN)
ax1.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v,p: f'${v/1000:.0f}B'))
ax2.yaxis.set_major_formatter(mtick.PercentFormatter())
ax1.set_xticks(YEARS)
ax1.set_title('AbbVie Revenue Trend 2020–2023', fontsize=14, fontweight='bold', pad=15)
ax1.set_axisbelow(True); ax1.yaxis.grid(True)
plt.tight_layout()
plt.savefig('images/01_revenue_trend.png', bbox_inches='tight')
plt.show()

---
## Section 3: Profitability & Margin Analysis <a id='3'></a>

In [ ]:
# ── Calculate all margin ratios ──
gross_margin    = [gp/r*100 for gp,r in zip(IS['gross_profit'],     IS['revenue'])]
operating_margin= [oi/r*100 for oi,r in zip(IS['operating_income'], IS['revenue'])]
net_margin      = [ni/r*100 for ni,r in zip(IS['net_income'],        IS['revenue'])]
ebitda_margin   = [e/r*100  for e,r  in zip(IS['ebitda'],            IS['revenue'])]

margin_df = pd.DataFrame({
    'Year':            YEARS,
    'Gross Margin %':  [round(m,1) for m in gross_margin],
    'EBITDA Margin %': [round(m,1) for m in ebitda_margin],
    'Operating Margin%':[round(m,1) for m in operating_margin],
    'Net Margin %':    [round(m,1) for m in net_margin],
}).set_index('Year')

print('=== Margin Analysis ===')
print(margin_df.to_string())
print(f'\nAvg Gross Margin:    {sum(gross_margin)/len(gross_margin):.1f}%')
print(f'Avg EBITDA Margin:   {sum(ebitda_margin)/len(ebitda_margin):.1f}%')
print(f'Avg Operating Margin:{sum(operating_margin)/len(operating_margin):.1f}%')

---
## Section 9: Executive Summary & Recommendations <a id='9'></a>

## Key Findings

### 1. Revenue grew 27% from 2020 to 2022 before a controlled pullback in 2023
The Allergan acquisition drove significant revenue expansion. The 2023 dip to $55.4B reflects the anticipated Humira biosimilar competition — not operational weakness.

### 2. Gross margin held above 75% for all 4 years
AbbVie maintained exceptional pricing power across its portfolio. Even as revenue normalized in 2023, gross margin stayed at 75.1% — reflecting strong product mix and brand protection.

### 3. $28.5B in debt was eliminated in 3 years
Post-acquisition debt of $87.1B was aggressively reduced to $58.6B by 2023 — a 33% reduction funded entirely by operating cash flow. This is a major signal of balance sheet discipline.

### 4. $21B+ in free cash flow funds dividends and future acquisitions
AbbVie's FCF consistently exceeds $20B annually, comfortably covering $10.9B in dividends while leaving capital for pipeline investment and M&A.

### 5. EBITDA margin averaged 43% — best-in-class for pharma
For context, the pharma industry average EBITDA margin is 20–25%. AbbVie's 43% average reflects its portfolio of high-margin specialty drugs.

---

## Business Recommendations

| Priority | Finding | Recommendation |
|---|---|---|
| High | Humira revenue declining | Accelerate Skyrizi & Rinvoq as primary growth drivers |
| High | Debt still elevated at $58.6B | Continue deleveraging — target sub-$50B by 2025 |
| Medium | R&D spend declining as % of revenue | Reinvest FCF into pipeline to prevent long-term revenue cliff |
| Medium | Current ratio below 1.0x in some years | Monitor liquidity — FCF covers but leaves thin margin |
| Low | Dividend payout growing faster than FCF | Ensure payout ratio doesn't exceed 60% of FCF |

---
*Analysis by Darius Nobles | [LinkedIn](https://www.linkedin.com/in/dariusnobles/) | [Portfolio](https://dariusnobles.netlify.app/)*